# 1 Generating Synthetic Data for Niagara Falls, ON.

In [1]:
import pandas as pd
import numpy as np
import math

print("Starting Generating Synthetic Data for Niagara Falls, ON")
np.random.seed(42) # Ensure reproducibility

# 1. Geographic Settings: Niagara Falls, Ontario
# Bounding box roughly covering the tourist and residential areas
lat_min, lat_max = 43.05, 43.12
lon_min, lon_max = -79.13, -79.05

num_demand_nodes = 28
num_candidate_stations = 12

# 2. Generate Demand Nodes (Ext B & C: peak/off-peak split, priority weights)
demand_nodes = []
for i in range(num_demand_nodes):
    lat = np.random.uniform(lat_min, lat_max)
    lon = np.random.uniform(lon_min, lon_max)
    
    # Base daily demand (cars)
    base_demand = np.random.randint(10, 50)
    
    # Ext B: Peak / Off-peak split (Peak gets 70% of traffic)
    demand_peak = base_demand * 0.70
    demand_off_peak = base_demand * 0.30
    
    # Ext C: Priority Weights (1 = Normal, 3 = High Priority/Tourist/Hospital)
    # 20% of nodes are high priority
    priority = 3.0 if np.random.rand() > 0.8 else 1.0 
    
    demand_nodes.append({
        'Node_ID': i,
        'Latitude': lat,
        'Longitude': lon,
        'Demand_Peak': round(demand_peak, 2),
        'Demand_OffPeak': round(demand_off_peak, 2),
        'Priority_Weight_wi': priority
    })

df_demand = pd.DataFrame(demand_nodes)
df_demand.to_csv('niagara_demand_nodes.csv', index=False)
print(f"Generated {num_demand_nodes} demand nodes -> 'niagara_demand_nodes.csv'")

# 3. Generate Candidate Stations (Ext A & Grid Limits: Solar/Wind/Grid max capacities)
candidate_stations = []
for j in range(num_candidate_stations):
    lat = np.random.uniform(lat_min, lat_max)
    lon = np.random.uniform(lon_min, lon_max)
    
    # Setup cost (C_j)
    setup_cost = np.random.randint(40000, 80000)
    
    # Ext A: Renewable Capacities (kW)
    # Some stations have high solar, some have high wind based on location
    solar_max = np.random.randint(0, 150)
    wind_max = np.random.randint(0, 100)
    
    # NEW: Hard Grid Capacity Limit (kW) - Ext A reinforcement
    grid_max = np.random.randint(100, 300) # Local transformer limit
    
    candidate_stations.append({
        'Candidate_ID': j,
        'Latitude': lat,
        'Longitude': lon,
        'Setup_Cost': setup_cost,
        'Solar_Max_kW': solar_max,
        'Wind_Max_kW': wind_max,
        'Grid_Max_kW': grid_max
    })

df_candidates = pd.DataFrame(candidate_stations)
df_candidates.to_csv('niagara_candidate_stations.csv', index=False)
print(f"Generated {num_candidate_stations} candidate stations -> 'niagara_candidate_stations.csv'")

# 4. Generate Coverage Matrix (r_ij) based on ACTUAL geographic distance
# We use the Haversine formula to calculate real km between points
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

acceptable_radius_km = 3.5 # EVs are willing to drive up to 3.5 km to charge

coverage_data = []
for i, row_d in df_demand.iterrows():
    for j, row_c in df_candidates.iterrows():
        dist_km = haversine(row_d['Latitude'], row_d['Longitude'], row_c['Latitude'], row_c['Longitude'])
        
        # r_ij is 1 if distance <= radius, else 0
        r_ij = 1 if dist_km <= acceptable_radius_km else 0
        
        coverage_data.append({
            'Node_ID': i,
            'Candidate_ID': j,
            'Distance_km': round(dist_km, 2),
            'r_ij_Covered': r_ij
        })

df_coverage = pd.DataFrame(coverage_data)
df_coverage.to_csv('niagara_coverage_matrix.csv', index=False)
print("Generated Coverage Matrix based on actual distance -> 'niagara_coverage_matrix.csv'")
print("\nPhase 2 Complete! Data is perfectly aligned with Ext A, B, and C.")

Starting Generating Synthetic Data for Niagara Falls, ON
Generated 28 demand nodes -> 'niagara_demand_nodes.csv'
Generated 12 candidate stations -> 'niagara_candidate_stations.csv'
Generated Coverage Matrix based on actual distance -> 'niagara_coverage_matrix.csv'

Phase 2 Complete! Data is perfectly aligned with Ext A, B, and C.
